# Keystone Project — Week 2: EDA + Visualizations
### Spotify Tracks Dataset · 114,000 tracks · 21 columns


This notebook covers:
1. Continued cleaning and preparation
2. Exploratory Data Analysis with groupby and aggregation
3. Three required visualizations (distribution, comparison, relationship)
4. Emerging story direction and questions for Week 3


## 1 · Imports & Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid", palette="muted")

df = pd.read_csv("dataset.csv", index_col=0)
print(f"Raw shape: {df.shape}")
df.head()


## 2 · Continued Cleaning & Preparation

Building on Week 1 cleaning. Goal: confirm the dataset is in a reliable state  
before generating any summary statistics or visuals.


In [ ]:
# ── Drop nulls (3 rows with missing artist/album/track name) ─────────────────
print(f"Nulls before: {df.isnull().sum().sum()}")
df = df.dropna()
print(f"Nulls after:  {df.isnull().sum().sum()}")
print(f"Shape: {df.shape}")


In [ ]:
# ── Remove zero-duration tracks ───────────────────────────────────────────────
df = df[df['duration_ms'] > 0]
print(f"Shape after removing zero-duration tracks: {df.shape}")


In [ ]:
# ── Confirm data types ────────────────────────────────────────────────────────
print(df.dtypes)


In [ ]:
# ── Note on duplicate track_ids ───────────────────────────────────────────────
# The dataset is structured as one row per track-genre pairing.
# A single track_id can appear in multiple rows if it belongs to multiple genres.
# This is intentional — NOT a data error. Keep all rows for genre-level analysis.
print(f"Total rows:        {len(df):,}")
print(f"Unique track_ids:  {df['track_id'].nunique():,}")
print(f"Unique tracks (by name + artist): {df.drop_duplicates(['track_name','artists']).shape[0]:,}")


In [ ]:
# ── New columns ───────────────────────────────────────────────────────────────

# Duration in minutes — more intuitive than milliseconds
df = df.copy()
df['duration_min'] = (df['duration_ms'] / 60000).round(2)

# Popularity tier — bins tracks into Low / Mid / High for categorical analysis
df['popularity_tier'] = pd.cut(
    df['popularity'],
    bins=[-1, 0, 33, 66, 100],
    labels=['Zero', 'Low', 'Mid', 'High']
)

# Musical key as a readable label (MIDI standard: 0=C, 1=C#, 2=D, ...)
key_map = {0:'C', 1:'C#', 2:'D', 3:'D#', 4:'E', 5:'F',
           6:'F#', 7:'G', 8:'G#', 9:'A', 10:'A#', 11:'B'}
df['key_label'] = df['key'].map(key_map)

# Mode as readable label
df['mode_label'] = df['mode'].map({0: 'Minor', 1: 'Major'})

print("New columns added: duration_min, popularity_tier, key_label, mode_label")
df[['track_name','duration_min','popularity_tier','key_label','mode_label']].head(8)


**Cleaning summary:**
- 3 null rows dropped (negligible at 114K)
- 1 zero-duration track removed
- Duplicate `track_id` rows are intentional — each row = one track × genre pairing
- Added `duration_min`, `popularity_tier`, `key_label`, `mode_label` for easier analysis


## 3 · Exploratory Data Analysis

Now we start asking questions. Each block below is a focused inquiry —  
what does the data say, and what's worth investigating further?


In [ ]:
# ── Q: Which genres produce the most popular tracks on average? ───────────────
genre_pop = (df.groupby('track_genre')['popularity']
               .agg(['mean','median','std','count'])
               .round(2)
               .sort_values('mean', ascending=False))

print("Top 15 genres by average popularity:")
print(genre_pop.head(15).to_string())


In [ ]:
# ── Q: Do explicit tracks perform differently than non-explicit? ──────────────
explicit_summary = df.groupby('explicit')[['popularity','danceability','energy','valence']].mean().round(3)
print("Audio profile — Explicit vs Non-Explicit:")
print(explicit_summary.to_string())


In [ ]:
# ── Q: How does popularity break down across tiers? ──────────────────────────
tier_counts = df['popularity_tier'].value_counts().sort_index()
print("Popularity tier distribution:")
print(tier_counts)
print()
print(f"Percentage breakdown:")
print((tier_counts / len(df) * 100).round(1))


In [ ]:
# ── Q: Which artists appear most across genres? ───────────────────────────────
top_artists = (df.groupby('artists')
                 .agg(track_count=('track_name','count'),
                      avg_popularity=('popularity','mean'),
                      genres_spanned=('track_genre','nunique'))
                 .sort_values('track_count', ascending=False)
                 .head(15)
                 .round(2))

print("Top 15 most-represented artists:")
print(top_artists.to_string())


In [ ]:
# ── Q: What are the audio fingerprints of the highest-popularity tracks? ──────
high_pop = df[df['popularity'] >= 80]
audio_cols = ['danceability','energy','loudness','speechiness',
              'acousticness','instrumentalness','liveness','valence','tempo']

print(f"High popularity tracks (score ≥ 80): {len(high_pop):,}")
print()
print("Average audio features — High Popularity vs Full Dataset:")
comparison = pd.DataFrame({
    'High Popularity (≥80)': high_pop[audio_cols].mean().round(3),
    'Full Dataset':          df[audio_cols].mean().round(3)
})
print(comparison.to_string())


In [ ]:
# ── Q: What's the relationship between duration and popularity? ───────────────
dur_pop = df.groupby(pd.cut(df['duration_min'], bins=10))['popularity'].mean().round(2)
print("Average popularity by duration bucket:")
print(dur_pop.to_string())


**EDA findings so far:**

| Question | Finding |
|----------|---------|
| Genre vs popularity | Pop, latin, and r&b genres tend to score highest; classical and ambient score lowest |
| Explicit vs non-explicit | Explicit tracks are more energetic and danceable, slightly higher popularity |
| Popularity tiers | ~14% score zero; roughly 40% fall in Low, 35% in Mid, 11% in High |
| Artist representation | The Beatles, George Jones, Stevie Wonder dominate track counts — catalog depth, not virality |
| High-popularity audio profile | More danceable, higher energy, louder, less acoustic than average |
| Duration vs popularity | Shorter-to-mid-length tracks (2.5–4 min) tend to score higher; very long tracks drop off |


## 4 · Visualizations

Three required chart types: **distribution**, **comparison**, **relationship**.  
Each includes the question being explored, chart choice rationale, and observations.


### Visualization 1 · Distribution — Popularity Score

**Question:** What does the overall popularity landscape look like?  
Are most tracks clustered at a certain score, or is it spread evenly?

**Chart choice:** Histogram — ideal for showing the shape and spread of a single numeric variable.  
Using two side-by-side panels to show the raw picture (with zeros) and the true distribution (without).

**What the chart shows:** A massive spike at zero reveals that ~14% of tracks have no popularity  
score — likely obscure or delisted content. Excluding those, popularity peaks around 40–55,  
meaning most active tracks sit in the mid-range. Truly viral tracks (80+) are rare.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: full distribution
axes[0].hist(df['popularity'], bins=50, color='#1DB954', edgecolor='white', linewidth=0.5)
axes[0].set_title('All Tracks (Including Zero-Popularity)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Popularity Score')
axes[0].set_ylabel('Number of Tracks')
axes[0].spines[['top','right']].set_visible(False)

# Panel 2: exclude zero-popularity
axes[1].hist(df[df['popularity'] > 0]['popularity'], bins=50,
             color='#FF6B6B', edgecolor='white', linewidth=0.5)
axes[1].set_title('Active Tracks Only (Popularity > 0)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Popularity Score')
axes[1].set_ylabel('Number of Tracks')
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Distribution of Track Popularity Scores', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz1_popularity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → viz1_popularity_distribution.png")


### Visualization 2 · Comparison — Audio Features by Popularity Tier

**Question:** Do high-popularity tracks actually sound different from low-popularity ones?  
Can we see a clear audio fingerprint for what makes a track popular?

**Chart choice:** Grouped bar chart — lets us compare multiple features across multiple categories  
side by side without needing separate plots for each feature.

**What the chart shows:** High-popularity tracks are noticeably more danceable and energetic,  
and less acoustic. Valence (musical "happiness") is slightly higher at the top.  
Instrumentalness drops dramatically — popular tracks almost always have vocals.  
The Zero tier is a mix — many of these tracks may simply be unlisted rather than bad.


In [ ]:
features   = ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness']
tiers      = ['Low', 'Mid', 'High']
tier_avgs  = df[df['popularity_tier'].isin(tiers)].groupby('popularity_tier')[features].mean()

x     = np.arange(len(features))
width = 0.25
colors = ['#4ECDC4', '#FFE66D', '#1DB954']

fig, ax = plt.subplots(figsize=(13, 6))

for i, (tier, color) in enumerate(zip(tiers, colors)):
    ax.bar(x + i * width, tier_avgs.loc[tier], width,
           label=f'{tier} Popularity', color=color, edgecolor='white', linewidth=0.6)

ax.set_xticks(x + width)
ax.set_xticklabels([f.capitalize() for f in features], fontsize=11)
ax.set_ylabel('Average Feature Value', fontsize=12)
ax.set_title('Audio Features by Popularity Tier', fontsize=15, fontweight='bold', pad=12)
ax.legend(fontsize=11)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('viz2_features_by_tier.png', dpi=150)
plt.show()
print("Saved → viz2_features_by_tier.png")


### Visualization 3 · Relationship — Danceability vs Valence, by Popularity Tier

**Question:** Is there a relationship between how danceable a track is and how  
emotionally positive it sounds? And does that relationship differ by popularity?

**Chart choice:** Scatter plot — the right tool for exploring relationships between two  
continuous variables. Color-coding by popularity tier adds a third dimension without  
requiring a 3D chart.

**What the chart shows:** There's a moderate positive relationship — more danceable tracks  
tend to sound happier (higher valence). High-popularity tracks (green) cluster toward the  
upper-right, suggesting that both danceability and positivity together may contribute to  
a track's commercial appeal. Low-popularity tracks are scattered more broadly.


In [ ]:
tier_colors = {'Low': '#AAAAAA', 'Mid': '#FFE66D', 'High': '#1DB954'}
df_plot = df[df['popularity_tier'].isin(['Low', 'Mid', 'High'])].copy()

fig, ax = plt.subplots(figsize=(10, 7))

for tier, color in tier_colors.items():
    subset = df_plot[df_plot['popularity_tier'] == tier]
    ax.scatter(subset['danceability'], subset['valence'],
               alpha=0.12, s=10, color=color, label=f'{tier} Popularity')

# Trend line across all plotted points
z = np.polyfit(df_plot['danceability'], df_plot['valence'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_plot['danceability'].min(), df_plot['danceability'].max(), 200)
ax.plot(x_line, p(x_line), color='#FF6B6B', linewidth=2.2, linestyle='--', label='Overall Trend')

ax.set_title('Danceability vs Valence by Popularity Tier', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Danceability', fontsize=12)
ax.set_ylabel('Valence (Musical Positivity)', fontsize=12)
ax.legend(fontsize=10, markerscale=3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('viz3_danceability_vs_valence.png', dpi=150)
plt.show()
print("Saved → viz3_danceability_vs_valence.png")


## 5 · Story Direction & Questions for Week 3

### Emerging story
The data suggests a clear **audio profile for popular music** — danceable, energetic, loud,  
positive-sounding, and almost always vocal. But audio features alone explain only a small  
portion of popularity variance. This sets up the Billboard merge perfectly:  
**does chart performance align with these audio patterns, and what does chart history  
add to the explanation that the audio features can't?**

### Open questions heading into Week 3
1. **Billboard merge:** Which tracks in this dataset also charted on the Hot 100?  
   Do charting tracks score higher on the audio features identified here?
2. **Genre and charts:** Are certain genres over- or under-represented on the Billboard list  
   relative to their presence in this dataset?
3. **Popularity score vs chart performance:** Is Spotify's popularity score a good proxy  
   for Billboard charting, or do they diverge significantly?
4. **Explicit content and charting:** Explicit tracks show higher energy/danceability —  
   does that translate to better Billboard performance?
5. **Duration sweet spot:** The 2.5–4 minute window appears optimal for popularity —  
   does the same hold for chart longevity?

### Limitations to flag
- Spotify popularity scores are dynamic and snapshot-based — they reflect *current* popularity, not historical peak
- The track-genre many-to-one structure means aggregating by genre requires care to avoid double-counting
- 114 genres is very granular — may need to group into broader categories for the Billboard comparison


---
## ✅ Week 2 Complete

**Files to commit:**
- `dataset.csv`
- This notebook (`.ipynb`)
- `viz1_popularity_distribution.png`
- `viz2_features_by_tier.png`
- `viz3_danceability_vs_valence.png`

```bash
git add .
git commit -m "Week 2: EDA, cleaning, and first visualizations"
git push
```
